In [0]:
%run ./Parametros

In [0]:
from pyspark.sql.utils import AnalysisException
from pyspark.sql import SparkSession
from pyspark.dbutils import DBUtils
from pyspark.sql.functions import col, to_timestamp, lit
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from datetime import datetime, timedelta
from functools import reduce
from pyspark.sql.window import Window


In [0]:
def load_tables(dict_load_tables, catalog=dw_catalog):
    """
    Carga tablas desde el catálogo y esquemas especificados.

    Parámetros:
        dict_load_tables: diccionario que define las tablas a cargar (<esquema> : [<tabla>])

    Retorna:
        df: diccionario de DataFrames cargados
    """
    df = {}
    success = 0
    fails = 0
    print("<", " TABLES LOADED ".center(80, "="), ">")
    print(f" < FROM '{catalog}' CATALOG > ".center(84, " "))
    for schema in dict_load_tables:
        print(f" - '{schema}' SCHEMA :")
        for table_name in dict_load_tables[schema]:
            try:
                full_table_name = f"{catalog}.{schema}.{table_name}"
                df[table_name] = spark.table(full_table_name)
                print(f"\t└ [SUCCESS] '{table_name}' loaded.")
                success += 1
            except AnalysisException as e:
                print(f"\t└ [FAIL] '{table_name}', TypeError: ", e.getErrorClass())
                fails += 1
        print("")
    print(
        "<",
        f" Load tables finished. {success} tables loaded, {fails} failed, {success+fails} total ".center(
            80, "="
        ),
        ">",
    )
    return df


In [0]:
def create_widgets(widgets):
    """
    Iterar sobre la lista de nombres de widgets.
    Crear un widget de texto con el nombre dado, valor por defecto como cadena vacía y el mismo nombre como etiqueta.
      Parámetros:
      - widgets: lista de nombres de widgets
      
    """
    for w_name in widgets:
        dbutils.widgets.text(w_name, "", w_name)

In [0]:
def get_widget_values(widgets):
    """
    Iterar sobre la lista de nombres de widgets.
    Obtener el valor del widget y almacenarlo en el diccionario.
        Parámetros:
        - widgets: lista de nombres de widgets
        Retorna:
            Diccionario que contiene los nombres de los widgets y sus valores
            
    """
    dict_parameters = dict()
    print("<"," GET WIDGETS VALUES ".center(80,"="),">")
    print("<  Widget Name :  Value >".center(84," "))
    for w_name in widgets:
        try:
            dict_parameters[w_name] = dbutils.widgets.get(w_name)
            print(f" └ [SUCCESS] '{w_name}' : {repr(dict_parameters[w_name])}")
        except Exception as e:
            print(f" └ [FAIL] '{w_name}' : ERROR.")
    print("<","="*80,">")
    return dict_parameters

In [0]:
def get_dimensions_description(dfs, param):
    parameters = dict()
    Escenario = param.get("Escenario")
    print("<", " ADICIÓN DE DESCRIPCIONES DE DIMENSIÓN ".center(80, "="), ">")
    print("<  Nombre del parámetro : Valor >".center(84, " "))
    if Escenario:
        try:
            scenario = (
                dfs["Dim_Escenario"]
                .filter(col("id_escenario") == int(Escenario))
                .first()
            )
            parameters["ScenarioName"] = scenario.escenario_nombre
            print(" └ [ÉXITO] 'Nombre de Escenario' :", repr(parameters["ScenarioName"]),"\n")
        except:
            print(f" └ [ERROR] No fue posible obtener el nombre del escenario.")
    print("<","="*80,">")
    return parameters

In [0]:
def read_table(table_name: str):
    try:
        df = spark.table(table_name)
        print(f"[SUCCESS] Table loaded: {table_name}")
        return df
    except Exception as e:
        print(f"[ERROR] {str(e)}")
        return None

In [0]:
def reindexar_fact_kpi(table_name: str):

    df = spark.table(table_name)

    w = Window.orderBy(
        F.col("Id_Tiempo").asc(),
        F.col("Id_Kpi").asc(),
        F.col("Id_Escenario").asc()
    )
    df = df.withColumn(
        "Id_Fact_Kpi",
        F.row_number().over(w)
    )

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )